# 04 — Test Dataset, Feature Builder, and PK Sampler

Run before training. Expected batch shape: `B x 10 x 60 x 17`.

In [1]:
from pathlib import Path
import os, random, time, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

EXP_DIR = Path('/media/wadud/DriveUbuntu/GaitRecognition 2.0')
POSE_TAG = None  # None = auto-detect folder under data/poses with most .npz files
POSES_DIR = EXP_DIR / 'data' / 'poses'
SPLIT_DIR = EXP_DIR / 'data' / 'splits'
REPORT_DIR = EXP_DIR / 'data' / 'reports'
RESULT_DIR = EXP_DIR / 'results'
CHECKPOINT_DIR = EXP_DIR / 'checkpoints'
LOG_DIR = EXP_DIR / 'logs'
for d in [SPLIT_DIR, REPORT_DIR, RESULT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def detect_pose_root(poses_dir=POSES_DIR, pose_tag=POSE_TAG):
    if pose_tag is not None:
        root = poses_dir / pose_tag
        if not root.exists():
            raise FileNotFoundError(root)
        return root
    candidates = [p for p in poses_dir.iterdir() if p.is_dir()] if poses_dir.exists() else []
    if not candidates:
        raise FileNotFoundError(f'No pose folders under {poses_dir}')
    counts = sorted([(len(list(p.rglob('*.npz'))), p) for p in candidates], reverse=True, key=lambda x: x[0])
    return counts[0][1]

POSE_ROOT = detect_pose_root()
print('EXP_DIR  :', EXP_DIR)
print('POSE_ROOT:', POSE_ROOT)

SPLIT_NAME="LT"
SEQ_LEN=60
P=8
K=4
NUM_WORKERS=2
TRAIN_CSV=SPLIT_DIR/f"train_{SPLIT_NAME}.csv"
assert TRAIN_CSV.exists(), TRAIN_CSV

EXP_DIR  : /media/wadud/DriveUbuntu/GaitRecognition 2.0
POSE_ROOT: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/poses/yolo26l_pose


In [2]:
COCO_PARENTS = np.array([0,0,0,1,2,0,0,5,6,7,8,5,6,11,12,13,14], dtype=np.int64)
LEFT_RIGHT_PAIRS = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]

def crop_or_pad_sequence(X, seq_len=60, random_crop=True):
    T = X.shape[0]
    if T == seq_len:
        return X
    if T > seq_len:
        start = np.random.randint(0, T-seq_len+1) if random_crop else max(0, (T-seq_len)//2)
        return X[start:start+seq_len]
    pad = np.repeat(X[-1:], seq_len-T, axis=0)
    return np.concatenate([X, pad], axis=0)

def swap_left_right(X):
    X = X.copy()
    for l, r in LEFT_RIGHT_PAIRS:
        X[:, [l, r], :] = X[:, [r, l], :]
    return X

def augment_skeleton(X):
    if np.random.rand() < 0.5:
        X = swap_left_right(X)
    X = X + np.random.normal(0, 0.005, size=X.shape).astype(np.float32)
    X = X + np.random.normal(0, 0.01, size=(1,1,2)).astype(np.float32)
    return X

def build_gaittr_features(X):
    X = X.astype(np.float32)
    assert X.ndim == 3 and X.shape[1:] == (17,2), f'Bad skeleton shape: {X.shape}'
    joint = X.copy()
    joint_rel = X - X[:, 0:1, :]
    v1 = np.zeros_like(X); v1[:-1] = X[1:] - X[:-1]
    v2 = np.zeros_like(X); v2[:-2] = X[2:] - X[:-2]
    bone = X - X[:, COCO_PARENTS, :]
    feat = np.concatenate([joint, joint_rel, v1, v2, bone], axis=-1)
    return feat.transpose(2,0,1).astype(np.float32)  # 10,T,17

In [3]:
from torch.utils.data import Dataset, DataLoader, Sampler

class CASIABGaitTRDataset(Dataset):
    def __init__(self, df, label_map, seq_len=60, train=True):
        self.df = df.reset_index(drop=True)
        self.label_map = label_map
        self.seq_len = seq_len
        self.train = train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        data = np.load(row['pose_path'])
        X = data['keypoints_norm_filled'].astype(np.float32)
        X = crop_or_pad_sequence(X, self.seq_len, random_crop=self.train)
        if self.train:
            X = augment_skeleton(X)
        feat = build_gaittr_features(X)
        return {
            'x': torch.from_numpy(feat),
            'label': torch.tensor(self.label_map[str(row['subject'])], dtype=torch.long),
            'subject': str(row['subject']),
            'condition': str(row['condition']),
            'seq': str(row['seq']),
            'view': str(row['view']),
            'pose_path': str(row['pose_path'])
        }

class PKBatchSampler(Sampler):
    def __init__(self, labels, p=8, k=4, steps_per_epoch=1000):
        self.labels = np.asarray(labels); self.p=p; self.k=k; self.steps_per_epoch=steps_per_epoch
        self.label_to_indices = {}
        for idx, lab in enumerate(self.labels):
            self.label_to_indices.setdefault(int(lab), []).append(idx)
        self.unique_labels = list(self.label_to_indices.keys())
        if len(self.unique_labels) < p:
            raise ValueError(f'Need at least p={p} identities, got {len(self.unique_labels)}')
    def __iter__(self):
        for _ in range(self.steps_per_epoch):
            selected = random.sample(self.unique_labels, self.p)
            batch=[]
            for lab in selected:
                idxs = self.label_to_indices[lab]
                batch.extend(random.sample(idxs, self.k) if len(idxs) >= self.k else random.choices(idxs, k=self.k))
            yield batch
    def __len__(self): return self.steps_per_epoch

In [4]:
df_train=pd.read_csv(TRAIN_CSV); subjects=sorted(df_train.subject.astype(str).unique()); label_map={s:i for i,s in enumerate(subjects)}
print('samples:',len(df_train),'subjects:',len(subjects)); display(df_train.head())
ds=CASIABGaitTRDataset(df_train,label_map,SEQ_LEN,train=True); sample=ds[0]
print('single shape:',sample['x'].shape); assert sample['x'].shape==(10,60,17)
labels=[label_map[str(s)] for s in df_train.subject]
sampler=PKBatchSampler(labels,p=P,k=K,steps_per_epoch=5)
loader=DataLoader(ds,batch_sampler=sampler,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
batch=next(iter(loader)); print('batch shape:',batch['x'].shape,'unique ids:',torch.unique(batch['label']).numel())
assert batch['x'].shape==(P*K,10,60,17)
print('[OK] Ready for training')

samples: 8140 subjects: 74


,pose_path,subject,condition,seq,view
0,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,1,bg,1,0
1,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,1,bg,1,18
2,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,1,bg,1,36
3,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,1,bg,1,54
4,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,1,bg,1,72


single shape: torch.Size([10, 60, 17])
batch shape: torch.Size([32, 10, 60, 17]) unique ids: 8
[OK] Ready for training
